# 05 — Level 2: Counterfactual Fairness (Kusner et al., 2017)

**Author:** Matteo

**Question:** *"If this employee's protected attribute were different, would the
model predict the same outcome?"*

We take each employee, flip a protected attribute (Gender; and Age band),
re-run the model, and count how often the prediction changes. A high flip-rate
means the model's decision **depends on the protected attribute** — a violation
of counterfactual fairness.

Reuses the wrappers from `utils.py`.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import utils
from utils import (
    ScaledModelWrapper, load_artifacts, load_raw_with_split, AGE_THRESHOLD,
)

BASE_PATH = os.path.abspath(os.path.join(os.path.dirname("__file__"), ".."))
RF_DIR    = os.path.join(BASE_PATH, "data", "models", "RF")
XGB_DIR   = os.path.join(BASE_PATH, "data", "models", "XGBoost")
OUT_DIR   = os.path.join(BASE_PATH, "data", "fairness")
os.makedirs(OUT_DIR, exist_ok=True)
THRESHOLD = 0.20

In [ ]:
df_raw, df_encoded, feature_cols = load_raw_with_split()
rf  = load_artifacts(RF_DIR)
xgb = load_artifacts(XGB_DIR)
feature_names = rf["feature_names"]
rf_wrapper  = ScaledModelWrapper(rf["model"],  rf["scaler"],  feature_names)
xgb_wrapper = ScaledModelWrapper(xgb["model"], xgb["scaler"], feature_names)

## Counterfactual-fairness flip test

`Gender` is encoded as 0/1. We flip it to the opposite value and re-predict.
For Age we move each employee across the age threshold by a fixed offset,
keeping everything else constant.

In [ ]:
def cf_fairness_flip(wrapper, df_encoded, feature_names, attr,
                     flip_fn, threshold=THRESHOLD):
    """
    flip_fn(X_df) -> X_df_modified : returns a copy with `attr` perturbed.
    Returns flip_rate and a boolean mask of which employees flipped.
    """
    X = df_encoded[feature_names].astype("float64").copy()
    base_pred = (wrapper.predict_proba(X)[:, 1] >= threshold).astype(int)

    X_flip = flip_fn(X.copy())
    flip_pred = (wrapper.predict_proba(X_flip)[:, 1] >= threshold).astype(int)

    flipped = base_pred != flip_pred
    return flipped.mean(), flipped


def flip_gender(X):
    # Gender encoded 0/1 -> swap to the other class
    X["Gender"] = 1 - X["Gender"]
    return X


def shift_age(X, offset=15):
    # Push everyone across the age band (clipped to a realistic 18-60 range)
    X["Age"] = np.clip(X["Age"] + np.where(X["Age"] < AGE_THRESHOLD, offset, -offset),
                       18, 60)
    return X

In [ ]:
results = []
for model_name, wrapper in [("Random Forest", rf_wrapper), ("XGBoost", xgb_wrapper)]:
    g_rate, g_mask = cf_fairness_flip(wrapper, df_encoded, feature_names, "Gender", flip_gender)
    a_rate, a_mask = cf_fairness_flip(wrapper, df_encoded, feature_names, "Age", shift_age)
    results.append({"model": model_name,
                    "gender_flip_rate": round(g_rate, 3),
                    "age_flip_rate":    round(a_rate, 3)})
    print(f"{model_name}: flipping Gender changes {g_rate:.1%} of predictions; "
          f"shifting Age changes {a_rate:.1%}.")

cf_fairness_df = pd.DataFrame(results)
cf_fairness_df.to_csv(os.path.join(OUT_DIR, "level2_counterfactual_fairness.csv"), index=False)

## Visualise flip rates

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
x = np.arange(len(cf_fairness_df))
ax.bar(x - 0.2, cf_fairness_df["gender_flip_rate"], 0.4, label="Flip Gender", color="#9C27B0")
ax.bar(x + 0.2, cf_fairness_df["age_flip_rate"],    0.4, label="Shift Age",   color="#009688")
ax.set_xticks(x); ax.set_xticklabels(cf_fairness_df["model"])
ax.set_ylabel("Prediction flip rate")
ax.set_title("Level 2 — Counterfactual Fairness (lower = more fair)")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "level2_counterfactual_fairness.png"), dpi=120)
plt.show()

### Reading Level 2
A non-trivial flip rate means the model relies on the protected attribute, even
indirectly through correlated features. This complements Level 1: parity can
look fine while the model is still counterfactually unfair. Note (Kusner 2017):
a fully causal treatment needs a structural model; our flip test is the
standard practical approximation used when no causal graph is available.